# 2. Outlier Detection and Treatment

Outliers are data points that differ significantly from other observations. This notebook covers:
- Z-score method
- IQR (Interquartile Range) method
- Multivariate detection with Mahalanobis distance
- Isolation Forest (machine learning approach)
- Treatment strategies: removal, capping, log transformation

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import mstats
from scipy.spatial.distance import mahalanobis
from numpy.linalg import inv
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
import warnings
warnings.filterwarnings('ignore')

## 2.1 Load the Data

In [ ]:
url = ("https://raw.githubusercontent.com/datasciencedojo/"
       "datasets/master/titanic.csv")
df = pd.read_csv(url)
print(f"Shape: {df.shape}")
print(f"\nFare statistics:")
print(df['Fare'].describe())

## 2.2 Z-Score Method

The **z-score** measures how many standard deviations a value is from the mean. Points with $|z| > 3$ are typically flagged as outliers. This works well for approximately normal distributions.

In [ ]:
def detect_zscore_outliers(series, threshold=3):
    z = np.abs(stats.zscore(series.dropna()))
    return z > threshold

for col in ['Age', 'Fare', 'SibSp', 'Parch']:
    data = df[col].dropna()
    outliers = detect_zscore_outliers(data)
    print(f"{col}: {outliers.sum()} outliers ({outliers.sum()/len(data)*100:.1f}%)")

## 2.3 IQR Method

The **IQR method** is robust to non-normal distributions. Outliers are defined as:
$$x < Q_1 - 1.5 \cdot IQR \quad \text{or} \quad x > Q_3 + 1.5 \cdot IQR$$

In [ ]:
def iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

num_cols = ['Age', 'Fare', 'SibSp', 'Parch']
for col in num_cols:
    data = df[col].dropna()
    lo, hi = iqr_bounds(data)
    n_out = ((data < lo) | (data > hi)).sum()
    print(f"{col}: {n_out} outliers, bounds=[{lo:.2f}, {hi:.2f}]")

## 2.4 Z-Score vs IQR on Skewed Data

When data is **skewed**, the z-score method (based on mean/std) underestimates outliers. The IQR method, based on percentiles, is more robust.

In [ ]:
fare = df['Fare'].dropna()
print(f"Fare skewness: {fare.skew():.2f}")

z_outliers = detect_zscore_outliers(fare).sum()
lo, hi = iqr_bounds(fare)
iqr_outliers = ((fare < lo) | (fare > hi)).sum()

print(f"Z-score outliers (|z| > 3): {z_outliers}")
print(f"IQR outliers: {iqr_outliers}")

## 2.5 Multivariate Detection: Mahalanobis Distance

**Mahalanobis distance** accounts for correlations between variables. A point can be an outlier in multivariate space even if it appears normal in each dimension individually.

In [ ]:
cols_mv = ['Age', 'Fare']
subset = df[cols_mv].dropna()
mean = subset.mean().values
cov_inv = inv(subset.cov().values)

distances = subset.apply(
    lambda row: mahalanobis(row.values, mean, cov_inv), axis=1)
mask = distances > 3
print(f"Multivariate outliers (Age, Fare): {mask.sum()} ({mask.sum()/len(mask)*100:.1f}%)")

## 2.6 Isolation Forest

**Isolation Forest** is a tree-based anomaly detection algorithm. It isolates outliers by randomly partitioning features. Outliers require fewer splits to isolate, so they have shorter path lengths.

In [ ]:
iso_data = df[['Age', 'Fare']].dropna()
iso = IsolationForest(contamination=0.05, random_state=42)
labels = iso.fit_predict(iso_data)

n_outliers = (labels == -1).sum()
print(f"Isolation Forest outliers: {n_outliers} ({n_outliers/len(iso_data)*100:.1f}%)")

## 2.7 Treatment Strategies

Once outliers are detected, we can:
1. **Remove** them (risk: losing real information)
2. **Cap/Winsorize** (clip to bounds)
3. **Log-transform** (reduce skewness)
4. **Robust scaling** (use median/IQR instead of mean/std)

In [ ]:
# Capping (winsorize)
df_capped = df.copy()
df_capped['Fare_capped'] = mstats.winsorize(df_capped['Fare'].fillna(0), limits=[0.01, 0.01])
print(f"Capping: max = {df_capped['Fare_capped'].max():.2f} (was {df['Fare'].max():.2f})")

# Log transform
df_log = df.copy()
df_log['Fare_log'] = np.log1p(df_log['Fare'])
print(f"Log: skew = {df_log['Fare_log'].skew():.2f} (was {df['Fare'].skew():.2f})")

# Robust scaling
scaler = RobustScaler()
fare_robust = scaler.fit_transform(df[['Fare']].fillna(0))
print(f"Robust scaling: median of scaled = {np.median(fare_robust):.4f}")

## Key Takeaways

| Method | Best For | Limitation |
|---|---|---|
| Z-score | Normal data | Fails on skewed data |
| IQR | Any distribution | Univariate only |
| Mahalanobis | Correlated features | Assumes elliptical distribution |
| Isolation Forest | Complex patterns | Black-box, needs tuning |